In [0]:
import os

# Read certs from Databricks Secrets
ca_cert      = dbutils.secrets.get("kafka-certs", "ca-cert")
service_cert = dbutils.secrets.get("kafka-certs", "service-cert")
service_key  = dbutils.secrets.get("kafka-certs", "service-key")

# Write certs to RAW external location
# This is an allowed path on Unity Catalog!
cert_base = "abfss://raw@boopathistorageaccount.dfs.core.windows.net/kafka-certs"

dbutils.fs.mkdirs(cert_base)

# Write using spark context
def write_to_adls(content, path):
    sc.parallelize([content], 1).saveAsTextFile(path)

# Write each cert as a single file
spark.createDataFrame([(ca_cert,)],      ["content"]).coalesce(1).write.mode("overwrite").text(f"{cert_base}/ca_cert")
spark.createDataFrame([(service_cert,)], ["content"]).coalesce(1).write.mode("overwrite").text(f"{cert_base}/service_cert")
spark.createDataFrame([(service_key,)],  ["content"]).coalesce(1).write.mode("overwrite").text(f"{cert_base}/service_key")

print("✅ Certs written to ADLS External Location!")
print(f"📁 Location: {cert_base}")

# Verify
files = dbutils.fs.ls(cert_base)
for f in files:
    print(f"  ✅ {f.name}")

In [0]:
# Read certs back from ADLS
def read_cert_from_adls(path):
    # Read the part file written by spark
    files = dbutils.fs.ls(path)
    part_file = [f.path for f in files if "part-" in f.name][0]
    return spark.read.text(part_file).collect()[0][0]

ca_cert_content      = read_cert_from_adls(f"{cert_base}/ca_cert")
service_cert_content = read_cert_from_adls(f"{cert_base}/service_cert")
service_key_content  = read_cert_from_adls(f"{cert_base}/service_key")

print(f"✅ ca.pem       : {len(ca_cert_content)} chars")
print(f"✅ service.cert : {len(service_cert_content)} chars")
print(f"✅ service.key  : {len(service_key_content)} chars")

In [0]:
from pyspark.sql.functions import from_json, col, current_timestamp, lit
from pyspark.sql.types import *

# ── Config ─────────────────────────────────────────────
BOOTSTRAP_SERVER = "fmcg-warehouse-mahendra-fmcg-warehouse-mahendra.d.aivencloud.com:25390"
TOPIC            = "warehouse.scan.events"

# ✅ Using your External Locations
CERT_PATH        = "abfss://raw@boopathistorageaccount.dfs.core.windows.net/kafka-certs"
CHECKPOINT_PATH  = "abfss://bronze@boopathistorageaccount.dfs.core.windows.net/checkpoints/bronze_scanner"
BRONZE_TABLE     = "bronze.scanner_events_raw"
# ────────────────────────────────────────────────────────

scanner_schema = StructType([
    StructField("event_id",       StringType(), True),
    StructField("warehouse_id",   StringType(), True),
    StructField("sku_id",         StringType(), True),
    StructField("event_type",     StringType(), True),
    StructField("quantity",       DoubleType(), True),
    StructField("scan_timestamp", StringType(), True),
    StructField("operator_id",    StringType(), True)
])

# ── Read from Aiven Kafka using External Location certs ─
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("kafka.security.protocol", "SSL") \
    .option("kafka.ssl.truststore.type", "PEM") \
    .option("kafka.ssl.truststore.location",
            f"{CERT_PATH}/ca_cert/part-00000") \
    .option("kafka.ssl.keystore.type", "PEM") \
    .option("kafka.ssl.keystore.location",
            f"{CERT_PATH}/service_cert/part-00000") \
    .option("kafka.ssl.keystore.key",
            dbutils.secrets.get("kafka-certs", "service-key")) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

print(f"✅ Kafka stream connected!")
print(f"📡 Bootstrap : {BOOTSTRAP_SERVER}")
print(f"📬 Topic     : {TOPIC}")

In [0]:
# Create databases
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# Parse Kafka JSON
df_parsed = df_raw \
    .withColumn("value", col("value").cast("string")) \
    .withColumn("data",  from_json(col("value"), scanner_schema)) \
    .select(
        "data.*",
        col("offset"),
        col("partition"),
        col("timestamp").alias("kafka_timestamp"),
        current_timestamp().alias("ingestion_timestamp"),
        lit("AIVEN_KAFKA").alias("source_name"),
        lit("BRONZE").alias("pipeline_layer")
    )

df_valid   = df_parsed.filter(col("event_id").isNotNull())
df_corrupt = df_parsed.filter(col("event_id").isNull())

# ✅ Write stream to Bronze Delta Table
query = df_valid.writeStream \
    .format("delta") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(BRONZE_TABLE)

print("🚀 Streaming pipeline started!")
print(f"📦 Writing to   : {BRONZE_TABLE}")
print(f"📁 Checkpoint   : {CHECKPOINT_PATH}")
print(f"🔒 Certs from   : {CERT_PATH}")
print("⏳ Events landing every 30 seconds...")

In [0]:
display(spark.sql("""
    SELECT 
        warehouse_id,
        event_type,
        COUNT(*)            as total_events,
        ROUND(AVG(quantity),2) as avg_quantity,
        MAX(ingestion_timestamp) as last_received
    FROM bronze.scanner_events_raw
    GROUP BY warehouse_id, event_type
    ORDER BY warehouse_id, event_type
"""))

In [0]:
display(spark.sql("SHOW EXTERNAL LOCATIONS"))

In [0]:
# Check what actually exists in raw container
try:
    files = dbutils.fs.ls("abfss://raw@boopathistorageaccount.dfs.core.windows.net/")
    print("📁 Contents of raw container:")
    for f in files:
        print(f"  → {f.name}  ({f.size} bytes)")
except Exception as e:
    print(f"❌ Error: {e}")

In [0]:
# Read from secrets
ca_cert      = dbutils.secrets.get("kafka-certs", "ca-cert")
service_cert = dbutils.secrets.get("kafka-certs", "service-cert")
service_key  = dbutils.secrets.get("kafka-certs", "service-key")

base = "abfss://raw@boopathistorageaccount.dfs.core.windows.net/kafka-certs"

# Write directly using dbutils.fs.put
dbutils.fs.put(f"{base}/ca.pem",       ca_cert,      overwrite=True)
dbutils.fs.put(f"{base}/service.cert", service_cert, overwrite=True)
dbutils.fs.put(f"{base}/service.key",  service_key,  overwrite=True)

print("✅ Certs written successfully!")

# Verify immediately
print("\n📁 Verifying files:")
for f in dbutils.fs.ls(base):
    print(f"  ✅ {f.name}  ({f.size} bytes)")


In [0]:
from pyspark.sql.functions import from_json, col, current_timestamp, lit
from pyspark.sql.types import *

# ── Config ──────────────────────────────────────────────
BOOTSTRAP_SERVER = "fmcg-warehouse-mahendra-fmcg-warehouse-mahendra.d.aivencloud.com:25390"
TOPIC            = "warehouse.scan.events"
CERT_BASE        = "abfss://raw@boopathistorageaccount.dfs.core.windows.net/kafka-certs"
CHECKPOINT_PATH  = "abfss://bronze@boopathistorageaccount.dfs.core.windows.net/checkpoints/bronze_scanner"
BRONZE_TABLE     = "bronze.scanner_events_raw"
# ────────────────────────────────────────────────────────

scanner_schema = StructType([
    StructField("event_id",       StringType(), True),
    StructField("warehouse_id",   StringType(), True),
    StructField("sku_id",         StringType(), True),
    StructField("event_type",     StringType(), True),
    StructField("quantity",       DoubleType(), True),
    StructField("scan_timestamp", StringType(), True),
    StructField("operator_id",    StringType(), True)
])

# ── Read from Aiven Kafka ────────────────────────────────
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("kafka.security.protocol", "SSL") \
    .option("kafka.ssl.truststore.type", "PEM") \
    .option("kafka.ssl.truststore.location", f"{CERT_BASE}/ca.pem") \
    .option("kafka.ssl.keystore.type", "PEM") \
    .option("kafka.ssl.keystore.location",   f"{CERT_BASE}/service.cert") \
    .option("kafka.ssl.keystore.key", dbutils.secrets.get("kafka-certs", "service-key")) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

# ── Parse JSON ───────────────────────────────────────────
df_parsed = df_raw \
    .withColumn("value", col("value").cast("string")) \
    .withColumn("data",  from_json(col("value"), scanner_schema)) \
    .select(
        "data.*",
        col("offset"),
        col("partition"),
        col("timestamp").alias("kafka_timestamp"),
        current_timestamp().alias("ingestion_timestamp"),
        lit("AIVEN_KAFKA").alias("source_name"),
        lit("BRONZE").alias("pipeline_layer")
    )

df_valid   = df_parsed.filter(col("event_id").isNotNull())
df_corrupt = df_parsed.filter(col("event_id").isNull())

# ── Create Databases ─────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# ── Start Streaming to Bronze Delta Table ────────────────
query = df_valid.writeStream \
    .format("delta") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(BRONZE_TABLE)

print("🚀 Streaming pipeline started!")
print(f"📡 Bootstrap : {BOOTSTRAP_SERVER}")
print(f"📬 Topic     : {TOPIC}")
print(f"📦 Table     : {BRONZE_TABLE}")
print(f"🔒 Certs     : {CERT_BASE}")
print(f"📁 Checkpoint: {CHECKPOINT_PATH}")
print("⏳ Events landing every 30 seconds...")


In [0]:
# Read both cert and key
service_cert = dbutils.secrets.get("kafka-certs", "service-cert")
service_key  = dbutils.secrets.get("kafka-certs", "service-key")

# Combine them into one file (Kafka requires this)
combined = service_key + "\n" + service_cert

base = "abfss://raw@boopathistorageaccount.dfs.core.windows.net/kafka-certs"

# Write combined file
dbutils.fs.put(f"{base}/client.pem", combined, overwrite=True)

print("✅ Combined cert+key written!")

# Verify all files
print("\n📁 All cert files:")
for f in dbutils.fs.ls(base):
    if f.size > 0:
        print(f"  ✅ {f.name}  ({f.size} bytes)")

In [0]:
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("kafka.security.protocol", "SSL") \
    .option("kafka.ssl.truststore.type", "PEM") \
    .option("kafka.ssl.truststore.location",
            "abfss://raw@boopathistorageaccount.dfs.core.windows.net/kafka-certs/ca.pem") \
    .option("kafka.ssl.keystore.type", "PEM") \
    .option("kafka.ssl.keystore.location",
            "abfss://raw@boopathistorageaccount.dfs.core.windows.net/kafka-certs/client.pem") \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

print("✅ Kafka stream reader created!")
print(f"📡 Bootstrap: {BOOTSTRAP_SERVER}")

In [0]:
from pyspark.sql.functions import from_json, col, current_timestamp, lit
from pyspark.sql.types import *

BOOTSTRAP_SERVER = "fmcg-warehouse-mahendra-fmcg-warehouse-mahendra.d.aivencloud.com:25390"
TOPIC            = "warehouse.scan.events"
CERT_BASE        = "abfss://raw@boopathistorageaccount.dfs.core.windows.net/kafka-certs"
CHECKPOINT_PATH  = "abfss://bronze@boopathistorageaccount.dfs.core.windows.net/checkpoints/bronze_scanner"
BRONZE_TABLE     = "bronze.scanner_events_raw"

scanner_schema = StructType([
    StructField("event_id",       StringType(), True),
    StructField("warehouse_id",   StringType(), True),
    StructField("sku_id",         StringType(), True),
    StructField("event_type",     StringType(), True),
    StructField("quantity",       DoubleType(), True),
    StructField("scan_timestamp", StringType(), True),
    StructField("operator_id",    StringType(), True)
])

df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("kafka.security.protocol", "SSL") \
    .option("kafka.ssl.truststore.type", "PEM") \
    .option("kafka.ssl.truststore.location", f"{CERT_BASE}/ca.pem") \
    .option("kafka.ssl.keystore.type", "PEM") \
    .option("kafka.ssl.keystore.location",   f"{CERT_BASE}/client.pem") \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

df_parsed = df_raw \
    .withColumn("value", col("value").cast("string")) \
    .withColumn("data",  from_json(col("value"), scanner_schema)) \
    .select(
        "data.*",
        col("offset"),
        col("partition"),
        col("timestamp").alias("kafka_timestamp"),
        current_timestamp().alias("ingestion_timestamp"),
        lit("AIVEN_KAFKA").alias("source_name"),
        lit("BRONZE").alias("pipeline_layer")
    )

df_valid = df_parsed.filter(col("event_id").isNotNull())

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

query = df_valid.writeStream \
    .format("delta") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(BRONZE_TABLE)

print("🚀 Streaming pipeline started!")
print(f"📡 Bootstrap : {BOOTSTRAP_SERVER}")
print(f"📬 Topic     : {TOPIC}")
print(f"📦 Table     : {BRONZE_TABLE}")
print("⏳ Events landing every 30 seconds...")

In [0]:
display(spark.sql("""
    SELECT 
        warehouse_id,
        event_type,
        COUNT(*)                 as total_events,
        ROUND(AVG(quantity), 2)  as avg_quantity,
        MAX(ingestion_timestamp) as last_received
    FROM bronze.scanner_events_raw
    GROUP BY warehouse_id, event_type
    ORDER BY warehouse_id, event_type
"""))

In [0]:
# Check if stream is actually running
for s in spark.streams.active:
    print(f"Stream ID   : {s.id}")
    print(f"Stream Name : {s.name}")
    print(f"Status      : {s.status}")
    print(f"Is Active   : {s.isActive}")
    print(f"Recent Progress: {s.recentProgress}")

In [0]:
from pyspark.sql.functions import from_json, col, current_timestamp, lit
from pyspark.sql.types import *

BOOTSTRAP_SERVER = "fmcg-warehouse-mahendra-fmcg-warehouse-mahendra.d.aivencloud.com:25390"
TOPIC            = "warehouse.scan.events"
CERT_BASE        = "abfss://raw@boopathistorageaccount.dfs.core.windows.net/kafka-certs"
CHECKPOINT_PATH  = "abfss://bronze@boopathistorageaccount.dfs.core.windows.net/checkpoints/bronze_scanner"
BRONZE_TABLE     = "bronze.scanner_events_raw"

scanner_schema = StructType([
    StructField("event_id",       StringType(), True),
    StructField("warehouse_id",   StringType(), True),
    StructField("sku_id",         StringType(), True),
    StructField("event_type",     StringType(), True),
    StructField("quantity",       DoubleType(), True),
    StructField("scan_timestamp", StringType(), True),
    StructField("operator_id",    StringType(), True)
])

df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("kafka.security.protocol", "SSL") \
    .option("kafka.ssl.truststore.type", "PEM") \
    .option("kafka.ssl.truststore.location", f"{CERT_BASE}/ca.pem") \
    .option("kafka.ssl.keystore.type", "PEM") \
    .option("kafka.ssl.keystore.location",   f"{CERT_BASE}/client.pem") \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

df_parsed = df_raw \
    .withColumn("value", col("value").cast("string")) \
    .withColumn("data",  from_json(col("value"), scanner_schema)) \
    .select(
        "data.*",
        col("offset"),
        col("partition"),
        col("timestamp").alias("kafka_timestamp"),
        current_timestamp().alias("ingestion_timestamp"),
        lit("AIVEN_KAFKA").alias("source_name"),
        lit("BRONZE").alias("pipeline_layer")
    )

df_valid = df_parsed.filter(col("event_id").isNotNull())

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

query = df_valid.writeStream \
    .format("delta") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(BRONZE_TABLE)

print("🚀 Stream running — waiting for data...")
print("⚠️  Do NOT stop this cell — open new cell to query data")

# This keeps the stream alive!
query.awaitTermination(timeout=300)  # runs for 5 minutes

In [0]:
display(spark.sql("""
    SELECT warehouse_id, event_type, 
           COUNT(*) as total_events,
           MAX(ingestion_timestamp) as last_received
    FROM bronze.scanner_events_raw
    GROUP BY warehouse_id, event_type
    ORDER BY warehouse_id
"""))